# 21. Ensemble Learning: AdaBoost (Adaptive Boosting)

## Algorithm Category
**Type**: Ensemble Learning - Classification/Regression  
**Complexity**: Medium  
**Use Case**: Sequential boosting that adaptively weights misclassified examples

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the AdaBoost algorithm and boosting principles
- Implement AdaBoost for classification and regression
- Understand how AdaBoost adaptively weights training examples
- Visualize how weak learners combine to form a strong classifier
- Tune hyperparameters (n_estimators, learning_rate)
- Apply AdaBoost to real-world problems

## Historical Context

AdaBoost was developed by Freund and Schapire in 1996:
- Freund, Y. & Schapire, R.E. (1996): "A decision-theoretic generalization of on-line learning"
- First practical boosting algorithm
- Won the Gödel Prize in 2003 for theoretical contributions

**Key Papers/References:**
- Freund, Y. & Schapire, R.E. (1996). "A decision-theoretic generalization of on-line learning"
- Freund, Y. & Schapire, R.E. (1997). "A short introduction to boosting"

## When to Use AdaBoost

AdaBoost is appropriate when:
- You have weak learners (slightly better than random)
- Working with binary or multiclass classification
- You want to improve performance iteratively
- Data has clear patterns that can be learned sequentially
- Interpretability of ensemble is helpful
- Moderate-sized datasets

## Theory & Mechanics

### Mathematical Foundation

AdaBoost combines weak learners sequentially, focusing on misclassified examples.

**Initialization:**
- Start with uniform weights: $w_i^{(1)} = \frac{1}{N}$ for all samples

**For each iteration t = 1, 2, ..., T:**

1. **Train weak learner**: $h_t(x)$ on weighted training data
2. **Calculate error**: $\epsilon_t = \sum_{i=1}^{N} w_i^{(t)} \mathbf{1}(h_t(x_i) \neq y_i)$
3. **Calculate learner weight**: $\alpha_t = \frac{1}{2}\ln\left(\frac{1-\epsilon_t}{\epsilon_t}\right)$
4. **Update sample weights**: 
   $$w_i^{(t+1)} = \frac{w_i^{(t)} \exp(-\alpha_t y_i h_t(x_i))}{Z_t}$$
   Where $Z_t$ is normalization factor

**Final Prediction:**
$$\hat{y} = \text{sign}\left(\sum_{t=1}^{T} \alpha_t h_t(x)\right)$$

### How It Works

1. **Start**: Train first weak learner on uniformly weighted data
2. **Evaluate**: Calculate error and assign weight to learner
3. **Re-weight**: Increase weights of misclassified examples
4. **Repeat**: Train next learner on re-weighted data
5. **Combine**: Weighted majority vote of all learners

### Key Hyperparameters

- **n_estimators**: Number of weak learners (boosting iterations)
- **learning_rate**: Shrinks contribution of each learner (default: 1.0)
- **base_estimator**: Type of weak learner (default: DecisionTreeClassifier with max_depth=1)
- **algorithm**: 'SAMME' or 'SAMME.R' (for multiclass)

### Advantages

- Simple to implement and understand
- No prior knowledge needed about weak learners
- Adaptive: focuses on hard examples
- Less prone to overfitting than single strong learner
- Works well with weak learners (stumps)

### Limitations

- Sensitive to noisy data and outliers
- Sequential training (cannot parallelize)
- May overfit with too many weak learners
- Requires careful tuning of learning rate


## Implementation

Let's implement AdaBoost for classification.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, mean_squared_error

# Import our helper functions
from src.models.supervised import split_data, evaluate_classifier, evaluate_regressor
from src.models.classification import calculate_classification_metrics, plot_confusion_matrix
from src.models.ensemble import extract_feature_importance
from src.utils.benchmarking import benchmark_model_training
from src.utils.validation import validate_model_output, check_cross_validation_stability

print("Libraries imported successfully!")


In [ ]:
# Load dataset - Breast Cancer
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target, name='Target')

print(f"Dataset Shape: {X.shape}")
print(f"Classes: {cancer.target_names.tolist()}")

# Split data
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

# Train AdaBoost Classifier
base_estimator = DecisionTreeClassifier(max_depth=1, random_state=42)  # Decision stumps
model = AdaBoostClassifier(
    base_estimator=base_estimator,
    n_estimators=50,
    learning_rate=1.0,
    random_state=42
)
model.fit(X_train, y_train)

print("\nAdaBoost Classifier:")
print(f"Number of estimators: {model.n_estimators}")
print(f"Learning rate: {model.learning_rate}")

# Make predictions
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nTest Accuracy: {accuracy:.3f}")

# Evaluate
results = evaluate_classifier(model, X_test, y_test)
metrics = calculate_classification_metrics(y_test.values, y_pred)
print(f"Precision: {metrics['precision']:.3f}, Recall: {metrics['recall']:.3f}, F1: {metrics['f1_score']:.3f}")


## Understanding Boosting Process

Let's visualize how AdaBoost improves over iterations.


In [ ]:
# Track performance over iterations
n_estimators_range = range(1, 101, 5)
train_scores = []
test_scores = []
estimator_errors = []

for n_est in n_estimators_range:
    ab = AdaBoostClassifier(
        base_estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        n_estimators=n_est,
        learning_rate=1.0,
        random_state=42
    )
    ab.fit(X_train, y_train)
    
    train_scores.append(accuracy_score(y_train, ab.predict(X_train)))
    test_scores.append(accuracy_score(y_test, ab.predict(X_test)))
    
    # Get average error of weak learners
    if hasattr(ab, 'estimator_errors_'):
        estimator_errors.append(np.mean(ab.estimator_errors_))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(n_estimators_range, train_scores, 'o-', label='Training Accuracy', markersize=3)
plt.plot(n_estimators_range, test_scores, 's-', label='Test Accuracy', markersize=3)
plt.xlabel('Number of Estimators')
plt.ylabel('Accuracy')
plt.title('AdaBoost: Performance vs Number of Estimators')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
if estimator_errors:
    plt.plot(n_estimators_range, estimator_errors, '^-', color='green', markersize=3)
    plt.xlabel('Number of Estimators')
    plt.ylabel('Average Weak Learner Error')
    plt.title('Weak Learner Error Over Iterations')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

optimal_n = n_estimators_range[np.argmax(test_scores)]
print(f"Optimal number of estimators: {optimal_n} with test accuracy: {max(test_scores):.3f}")


## Validation & Testing

Let's validate the model and compare with single decision tree.


In [ ]:
# Validation 1: Compare with single decision tree
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=1, random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
dt_accuracy = accuracy_score(y_test, dt_pred)

print("Comparison: Single Weak Learner vs AdaBoost")
print(f"  Single Decision Stump: {dt_accuracy:.3f}")
print(f"  AdaBoost (50 stumps): {accuracy:.3f}")
print(f"  Improvement: {accuracy - dt_accuracy:.3f}")

# Validation 2: Cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

print(f"\nCross-Validation Results (5-fold):")
print(f"  Mean Accuracy: {cv_mean:.3f} (+/- {cv_std:.3f})")

stability = check_cross_validation_stability(cv_scores, threshold=0.1)
print(f"  Is Stable: {stability['is_stable']}")

# Validation 3: Check model output
validation_result = validate_model_output(y_pred, y_test.values, task_type='classification')
assert validation_result['valid'], "Invalid predictions!"
assert accuracy > dt_accuracy, "AdaBoost should outperform single weak learner!"
print("\n✓ Validation checks passed")


## Feature Importance

Let's extract feature importance from AdaBoost.


In [ ]:
# Extract feature importance
feature_importance = extract_feature_importance(model, feature_names=X.columns.tolist())
print("Top 10 Most Important Features:")
print(feature_importance.head(10))

# Visualize
plt.figure(figsize=(10, 6))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['importance'], align='center')
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance')
plt.title('AdaBoost Feature Importance (Top 15)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## Effect of Learning Rate

Let's see how learning rate affects performance.


In [ ]:
# Test different learning rates
learning_rates = [0.1, 0.5, 1.0, 1.5, 2.0]
lr_train_scores = []
lr_test_scores = []

for lr in learning_rates:
    ab = AdaBoostClassifier(
        base_estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        n_estimators=50,
        learning_rate=lr,
        random_state=42
    )
    ab.fit(X_train, y_train)
    lr_train_scores.append(accuracy_score(y_train, ab.predict(X_train)))
    lr_test_scores.append(accuracy_score(y_test, ab.predict(X_test)))

plt.figure(figsize=(10, 6))
plt.plot(learning_rates, lr_train_scores, 'o-', label='Training Accuracy')
plt.plot(learning_rates, lr_test_scores, 's-', label='Test Accuracy')
plt.xlabel('Learning Rate')
plt.ylabel('Accuracy')
plt.title('AdaBoost: Effect of Learning Rate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

optimal_lr = learning_rates[np.argmax(lr_test_scores)]
print(f"Optimal learning rate: {optimal_lr} with test accuracy: {max(lr_test_scores):.3f}")


## Real-World Application

Let's tune hyperparameters and compare with other ensemble methods.


In [ ]:
# Hyperparameter tuning
param_grid = {
    'n_estimators': [25, 50, 100],
    'learning_rate': [0.5, 1.0, 1.5],
    'base_estimator__max_depth': [1, 2, 3]
}

grid_search = GridSearchCV(
    AdaBoostClassifier(
        base_estimator=DecisionTreeClassifier(random_state=42),
        random_state=42
    ),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print("\nBest Hyperparameters:")
print(grid_search.best_params_)
print(f"Best CV Accuracy: {grid_search.best_score_:.3f}")

# Compare with Random Forest
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)

print(f"\nComparison with Random Forest:")
print(f"  AdaBoost (tuned): {grid_search.best_score_:.3f} (CV)")
print(f"  Random Forest: {rf_accuracy:.3f} (test)")


## Summary & Key Takeaways

### Key Concepts Learned

1. **AdaBoost Basics**
   - Sequential ensemble method (boosting)
   - Combines weak learners into strong classifier
   - Adaptively weights misclassified examples
   - Each iteration focuses on previous mistakes

2. **Boosting Process**
   - Start with uniform weights
   - Train weak learner on weighted data
   - Calculate learner weight based on error
   - Re-weight samples (increase weight of misclassified)
   - Repeat until convergence

3. **Key Hyperparameters**
   - **n_estimators**: Number of boosting iterations
   - **learning_rate**: Shrinks contribution of each learner
   - **base_estimator**: Type of weak learner (usually decision stumps)

4. **Best Practices**
   - Use weak learners (decision stumps work well)
   - Tune learning rate to prevent overfitting
   - Monitor training vs test performance
   - Consider early stopping if overfitting occurs

### When to Use AdaBoost

✅ **Good for:**
- Binary and multiclass classification
- When you have weak learners available
- Moderate-sized datasets
- Clear patterns that can be learned sequentially
- When interpretability is helpful

❌ **Not ideal for:**
- Noisy data (sensitive to outliers)
- Very large datasets (sequential training is slow)
- When parallelization is important
- Regression tasks (use AdaBoostRegressor or Gradient Boosting)
- Real-time predictions (can be slow)

### Next Steps

- Try **Gradient Boosting** for more sophisticated boosting
- Explore **XGBoost** for optimized gradient boosting
- Compare with **Random Forest** (bagging vs boosting)
- Use **Early Stopping** to prevent overfitting
